# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and performing basic analyses on the [FAIR^2 dataset: "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All fields, record sets, and columns are referenced by their `@id` in line with the Croissant specification.

### Dataset Source
The dataset source is defined by a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata using standard attribute access
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n---\n{meta.description}\n")

## 2. Data Overview

List available record sets, their `@id`, and contents. Each record set contains a set of fields (which in turn map to columns in the underlying data).

We'll print all available record set `@id`s, their names, and sample the field `@id`s for each.

In [ ]:
# Display available record sets and their fields
record_sets = meta.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"  - RecordSet @id: '{rs.id}'  (name: '{rs.name}')")
    if hasattr(rs, 'fields'):
        print("    fields:")
        for field in rs.fields:
            print(f"      • Field @id: '{field.id}' (name: '{field.name}')")
    print()
if not record_sets:
    print("WARNING: No record sets found in the schema. Please verify data availability.")

## 3. Data Extraction

Load data for at least one record set into a pandas DataFrame. For each, use the record set and field `@id` as in the previous cell.

We show how to extract and inspect the columns for one record set (using `@id`), and preview the first few rows of data.

In [ ]:
# For this dataset, use the first available record set (by @id), if any
if record_sets:
    record_sets_ids = [rs.id for rs in record_sets]
    dataframes = {}
    for record_set_id in record_sets_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

    print(f"Record sets loaded: {list(dataframes.keys())}\n")
    first_rs = record_sets_ids[0]
    print(f"Columns in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets available for data extraction.")

## 4. Exploratory Data Analysis (EDA)

Here we demonstrate basic processing steps using `@id` field/column access as defined by the schema. Replace `<numeric_field_id>` and `<group_field_id>` below with valid field/column `@id`s from the record set you wish to analyze.

In [ ]:
# Example: Filter and normalize entries for a selected numeric field (by @id)
if record_sets:
    rs_id = record_sets_ids[0]
    df = dataframes[rs_id]

    # Choose numeric field @id from fields in the first record set
    # Use schema information for a likely numeric field. We'll try a safe guess:
    numeric_field_id = None
    group_field_id = None
    for field in record_sets[0].fields:
        # Heuristically pick the first numeric field or common names
        t = getattr(field, 'data_type', '')
        if t in ('schema:Float', 'schema:Number', 'schema:Integer', 'Float', 'Number', 'Integer') and numeric_field_id is None:
            numeric_field_id = field.id
        # Take a groupable field
        if not group_field_id:
            for kw in ['group', 'category', 'type', 'accession', 'description', 'sample']:
                if kw in field.id.lower() or kw in field.name.lower():
                    group_field_id = field.id

    print(f"Using numeric_field_id: {numeric_field_id}")
    print(f"Using group_field_id: {group_field_id}\n")

    # Proceed if numeric field exists in the DataFrame
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No suitable numeric field available for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if possible, its grouping. Adjust field @id as needed for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='purple')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

We have demonstrated how to load, explore, filter, and visualize data from a Croissant-standard FAIR^2 dataset using the `mlcroissant` library. More detailed analyses can reference specific record set, field, and column `@id`s for robust and reproducible data workflows.